# Análise de Jogadores FIFA 17

Notebook para carregar, diagnosticar, limpar e validar os dados. Execute as células na ordem apresentada.

## 1. Importar bibliotecas e configurar exibição

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 60)

## 2. Localizar e carregar o arquivo CSV

In [3]:
arquivo_csv = Path("FIFA+17+Players.csv")
df = pd.read_csv(arquivo_csv)

print(f"Arquivo carregado: {arquivo_csv.resolve()}")

Arquivo carregado: /workspaces/FinalProjectDataAnalyticsAndDecisionMaking/FIFA+17+Players.csv


## 3. Inspecionar dimensões, tipos e amostra dos dados

In [4]:
print("Dimensões:", df.shape)
df.info()
df.head()

Dimensões: (17591, 53)
<class 'pandas.DataFrame'>
RangeIndex: 17591 entries, 0 to 17590
Data columns (total 53 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                17591 non-null  str    
 1   Nationality         17589 non-null  str    
 2   National_Position   1077 non-null   str    
 3   National_Kit        1077 non-null   float64
 4   Club                17591 non-null  str    
 5   Club_Position       17590 non-null  str    
 6   Club_Kit            17590 non-null  float64
 7   Club_Joining        17590 non-null  str    
 8   Contract_Expiry     17590 non-null  float64
 9   Rating              17591 non-null  int64  
 10  Height              17591 non-null  str    
 11  Weight              17591 non-null  str    
 12  Preffered_Foot      17591 non-null  str    
 13  Birth_Date          17591 non-null  str    
 14  Age                 17591 non-null  int64  
 15  Preffered_Position  17591 non-null  str  

,Name,Nationality,National_Position,National_Kit,Club,Club_Position,Club_Kit,Club_Joining,Contract_Expiry,Rating,Height,Weight,Preffered_Foot,Birth_Date,Age,Preffered_Position,Work_Rate,Weak_foot,Skill_Moves,Ball_Control,Dribbling,Marking,Sliding_Tackle,Standing_Tackle,Aggression,Reactions,Attacking_Position,Interceptions,Vision,Composure,Crossing,Short_Pass,Long_Pass,Acceleration,Speed,Stamina,Strength,Balance,Agility,Jumping,Heading,Shot_Power,Finishing,Long_Shots,Curve,Freekick_Accuracy,Penalties,Volleys,GK_Positioning,GK_Diving,GK_Kicking,GK_Handling,GK_Reflexes
0,Cristiano Ronaldo,Portugal,LS,7.0,Real Madrid,LW,7.0,7/1/2009,2021.0,94,185 cm,80 kg,Right,2/5/1985,32,LW/ST,High / Low,4,5,93,92,22,23,31,63,96,94,29,85,86,84,83,77,91,92,92,80,63,90,95,85,92,93,90,81,76,85,88,14,7,15,11,11
1,Lionel Messi,Argentina,RW,10.0,FC Barcelona,RW,10.0,7/1/2004,2018.0,93,170 cm,72 kg,Left,6/24/1987,29,RW,Medium / Medium,4,4,95,97,13,26,28,48,95,93,22,90,94,77,88,87,92,87,74,59,95,90,68,71,85,95,88,89,90,74,85,14,6,15,11,8
2,Neymar,Brazil,LW,10.0,FC Barcelona,LW,11.0,7/1/2013,2021.0,92,174 cm,68 kg,Right,2/5/1992,25,LW,High / Medium,5,5,95,96,21,33,24,56,88,90,36,80,80,75,81,75,93,90,79,49,82,96,61,62,78,89,77,79,84,81,83,15,9,15,9,11
3,Luis Suárez,Uruguay,LS,9.0,FC Barcelona,ST,9.0,7/11/2014,2021.0,92,182 cm,85 kg,Right,1/24/1987,30,ST,High / Medium,4,4,91,86,30,38,45,78,93,92,41,84,83,77,83,64,88,77,89,76,60,86,69,77,87,94,86,86,84,85,88,33,27,31,25,37
4,Manuel Neuer,Germany,GK,1.0,FC Bayern,GK,1.0,7/1/2011,2021.0,92,193 cm,92 kg,Right,3/27/1986,31,GK,Medium / Medium,4,1,48,30,10,11,10,29,85,12,30,70,70,15,55,59,58,61,44,83,35,52,78,25,25,13,16,14,11,47,11,91,89,95,90,89


## 4. Diagnosticar valores nulos e duplicatas

In [5]:
n_linhas_antes = len(df)
nulos_antes = df.isna().sum().sum()
duplicatas = df.duplicated().sum()
nulos_por_coluna = df.isna().sum()

print("Linhas:", n_linhas_antes)
print("Total de nulos:", nulos_antes)
print("Linhas duplicadas:", duplicatas)
display(nulos_por_coluna[nulos_por_coluna > 0])

Linhas: 17591
Total de nulos: 33034
Linhas duplicadas: 3


Nationality              2
National_Position    16514
National_Kit         16514
Club_Position            1
Club_Kit                 1
Club_Joining             1
Contract_Expiry          1
dtype: int64

## 5. Remover duplicatas exatas

In [6]:
df = df.drop_duplicates()
linhas_removidas = n_linhas_antes - len(df)

print("Linhas removidas:", linhas_removidas)
print("Linhas após remoção:", len(df))

Linhas removidas: 3
Linhas após remoção: 17588


## 6. Extrair altura e peso em valores numéricos

In [7]:
df["Height_cm"] = df["Height"].str.extract(r"(\d+)").astype(float)
df["Weight_kg"] = df["Weight"].str.extract(r"(\d+)").astype(float)

df[["Height", "Height_cm", "Weight", "Weight_kg"]].head()

,Height,Height_cm,Weight,Weight_kg
0,185 cm,185.0,80 kg,80.0
1,170 cm,170.0,72 kg,72.0
2,174 cm,174.0,68 kg,68.0
3,182 cm,182.0,85 kg,85.0
4,193 cm,193.0,92 kg,92.0


## 7. Converter colunas de data

In [8]:
df["Birth_Date"] = pd.to_datetime(
    df["Birth_Date"], format="%m/%d/%Y", errors="coerce"
)
df["Club_Joining"] = pd.to_datetime(
    df["Club_Joining"], format="%m/%d/%Y", errors="coerce"
)

print(df[["Birth_Date", "Club_Joining"]].dtypes)
print(df[["Birth_Date", "Club_Joining"]].isna().sum())

Birth_Date      datetime64[us]
Club_Joining    datetime64[us]
dtype: object
Birth_Date      0
Club_Joining    1
dtype: int64


## 8. Criar indicador de convocação para seleção

In [9]:
df["Convocado_Selecao"] = df["National_Position"].notna()

df["Convocado_Selecao"].value_counts(dropna=False)

Convocado_Selecao
False    16513
True      1075
Name: count, dtype: int64

## 9. Separar taxa de trabalho em ataque e defesa

In [10]:
work_split = df["Work_Rate"].str.split(" / ", expand=True)
df["WorkRate_Ataque"] = work_split[0]
df["WorkRate_Defesa"] = work_split[1]

df[["Work_Rate", "WorkRate_Ataque", "WorkRate_Defesa"]].head()

,Work_Rate,WorkRate_Ataque,WorkRate_Defesa
0,High / Low,High,Low
1,Medium / Medium,Medium,Medium
2,High / Medium,High,Medium
3,High / Medium,High,Medium
4,Medium / Medium,Medium,Medium


## 10. Renomear colunas com grafia incorreta

In [11]:
df = df.rename(columns={
    "Preffered_Foot": "Preferred_Foot",
    "Preffered_Position": "Preferred_Position_Raw",
})

["Preferred_Foot", "Preferred_Position_Raw"]

['Preferred_Foot', 'Preferred_Position_Raw']

## 11. Preencher nacionalidades ausentes

In [12]:
df["Nationality"] = df["Nationality"].fillna("Desconhecido")

print("Nulos em Nationality:", df["Nationality"].isna().sum())

Nulos em Nationality: 0


## 11.1 Tratar jogador sem clube (nulos em Club_Position, Club_Kit, Club_Joining e Contract_Expiry)

In [15]:
colunas_clube = ["Club_Position", "Club_Kit", "Club_Joining", "Contract_Expiry"]
linhas_sem_clube = df[df[colunas_clube].isna().any(axis=1)]
display(linhas_sem_clube[["Name", "Club", *colunas_clube]])

# Confirmado: é a mesma linha (Didier Drogba, Club = "Free agent").
# Mantém-se a linha (jogador real, só sem clube) e marca-se Club_Position como
# "Free Agent". Club_Kit, Club_Joining e Contract_Expiry seguem nulos de propósito,
# pois não existe kit/data de contratação/expiração para quem não tem clube.
df["Club_Position"] = df["Club_Position"].fillna("Free Agent")

print("Nulos restantes em", colunas_clube, ":")
print(df[colunas_clube].isna().sum())

,Name,Club,Club_Position,Club_Kit,Club_Joining,Contract_Expiry
386,Didier Drogba,Free agent,Free Agent,NaN,NaT,NaN


Nulos restantes em ['Club_Position', 'Club_Kit', 'Club_Joining', 'Contract_Expiry'] :
Club_Position      0
Club_Kit           1
Club_Joining       1
Contract_Expiry    1
dtype: int64


## 11.2 Investigar outliers de idade (Age > 40)

In [17]:
df[df["Age"] > 40][["Name", "Age", "Club", "Preferred_Position_Raw"]]

,Name,Age,Club,Preferred_Position_Raw
2379,Essam El-Hadary,44,Free Agents,GK
4596,Óscar Pérez,44,Pachuca,GK
4634,Juan Sebastián Verón,42,Estudiantes,CM/CDM
5479,Dmitriy Loskov,43,Lokomotiv Moscow,CM/CAM
6506,Jussi Jääskeläinen,41,Wigan Athletic,GK
7471,Esteban,41,Real Oviedo,GK
10153,Néider Morantes,41,Boyacá Chicó,CAM/RM
10625,Roberto Colombo,41,Cagliari,GK
15050,Chris Day,41,Stevenage,GK
15202,Yukio Tsuchiya,42,Ventforet Kofu,CB


In [18]:
idosos = df[df["Age"] > 40]
print(f"Jogadores com Age > 40: {len(idosos)}")
print(idosos["Preferred_Position_Raw"].value_counts())

# Conclusão: caso legítimo, não outlier. Os nomes correspondem a jogadores reais
# (ex.: Essam El-Hadary, Óscar Pérez, Barry Richardson), majoritariamente goleiros,
# posição que costuma ter carreiras mais longas. Nenhuma linha foi removida ou alterada.

Jogadores com Age > 40: 15
Preferred_Position_Raw
GK        10
CM/CDM     1
CM/CAM     1
CAM/RM     1
CB         1
RM         1
Name: count, dtype: int64


## 12. Validar o resultado da limpeza

In [16]:
print(f"Linhas antes: {n_linhas_antes}")
print(f"Linhas depois: {len(df)}")
print(f"Total atual de nulos: {df.isna().sum().sum()}")
print(f"Duplicatas restantes: {df.duplicated().sum()}")

nulos_residuais = df.isna().sum()
print("Nulos residuais (esperados: Club_Kit, Club_Joining e Contract_Expiry")
print("do único jogador sem clube, Didier Drogba):")
print(nulos_residuais[nulos_residuais > 0])

colunas_transformadas = [
    "Height_cm",
    "Weight_kg",
    "Birth_Date",
    "Club_Joining",
    "Convocado_Selecao",
    "WorkRate_Ataque",
    "WorkRate_Defesa",
]
print(df[colunas_transformadas].dtypes)
display(df[["Name", *colunas_transformadas]].head())

Linhas antes: 17591
Linhas depois: 17588
Total atual de nulos: 33029
Duplicatas restantes: 0
Nulos residuais (esperados: Club_Kit, Club_Joining e Contract_Expiry
do único jogador sem clube, Didier Drogba):
National_Position    16513
National_Kit         16513
Club_Kit                 1
Club_Joining             1
Contract_Expiry          1
dtype: int64
Height_cm                   float64
Weight_kg                   float64
Birth_Date           datetime64[us]
Club_Joining         datetime64[us]
Convocado_Selecao              bool
WorkRate_Ataque                 str
WorkRate_Defesa                 str
dtype: object


,Name,Height_cm,Weight_kg,Birth_Date,Club_Joining,Convocado_Selecao,WorkRate_Ataque,WorkRate_Defesa
0,Cristiano Ronaldo,185.0,80.0,1985-02-05,2009-07-01,True,High,Low
1,Lionel Messi,170.0,72.0,1987-06-24,2004-07-01,True,Medium,Medium
2,Neymar,174.0,68.0,1992-02-05,2013-07-01,True,High,Medium
3,Luis Suárez,182.0,85.0,1987-01-24,2014-07-11,True,High,Medium
4,Manuel Neuer,193.0,92.0,1986-03-27,2011-07-01,True,Medium,Medium


## 12.1 Checagens de faixa (sanity checks)

In [19]:
assert df["Rating"].between(0, 100).all(), "Rating fora do intervalo"
assert df["Height_cm"].between(140, 210).all(), "Altura suspeita"
# Limite superior 47 (não 45): etapa 11.2 confirmou que goleiros com até 47 anos
# são casos reais, não erro de digitação.
assert df["Age"].between(15, 47).all(), "Idade suspeita"

print("Todas as checagens de faixa passaram.")

Todas as checagens de faixa passaram.


## 13. Agrupar posições em 4 categorias

In [20]:
mapa_posicao = {
    "GK": "Goleiro",
    "CB": "Defesa", "LB": "Defesa", "RB": "Defesa", "LWB": "Defesa", "RWB": "Defesa",
    "CDM": "Meio-campo", "CM": "Meio-campo", "CAM": "Meio-campo", "LM": "Meio-campo", "RM": "Meio-campo",
    "ST": "Ataque", "CF": "Ataque", "LW": "Ataque", "RW": "Ataque",
}

def agrupar_posicao(pos_raw):
    primeiro_token = str(pos_raw).split("/")[0].strip()
    return mapa_posicao.get(primeiro_token, "Outro")

df["Grupo_Posicao"] = df["Preferred_Position_Raw"].apply(agrupar_posicao)

print(df["Grupo_Posicao"].value_counts())
print("Não classificados:", (df["Grupo_Posicao"] == "Outro").sum())

Grupo_Posicao
Meio-campo    6558
Defesa        5749
Ataque        3278
Goleiro       2003
Name: count, dtype: int64
Não classificados: 0


## 14. Faixa etária, elite e situação contratual

In [21]:
df["Faixa_Idade"] = pd.cut(
    df["Age"], bins=[0, 21, 25, 29, 100],
    labels=["Até 21", "22-25", "26-29", "30+"]
)

df["Elite"] = df["Rating"] >= 80
df["Contrato_Acabando"] = df["Contract_Expiry"] <= 2017

print(df["Faixa_Idade"].value_counts().sort_index())
print("Jogadores elite (Rating>=80):", df["Elite"].sum())
print("Contrato acabando em 2017:", df["Contrato_Acabando"].sum())

Faixa_Idade
Até 21    4098
22-25     5382
26-29     4504
30+       3604
Name: count, dtype: int64
Jogadores elite (Rating>=80): 529
Contrato acabando em 2017: 4529


## 15. Continente a partir da nacionalidade

In [22]:
%pip install -q pycountry-convert

Note: you may need to restart the kernel to use updated packages.


In [23]:
import pycountry_convert as pc

excecoes_reino_unido = {
    "England": "Europe", "Scotland": "Europe", "Wales": "Europe",
    "Northern Ireland": "Europe", "Republic of Ireland": "Europe",
    "Kosovo": "Europe", "Desconhecido": "Desconhecido",
}

def obter_continente(pais):
    if pais in excecoes_reino_unido:
        return excecoes_reino_unido[pais]
    try:
        codigo = pc.country_name_to_country_alpha2(pais)
        continente_cod = pc.country_alpha2_to_continent_code(codigo)
        return pc.convert_continent_code_to_continent_name(continente_cod)
    except Exception:
        return "Não mapeado"

df["Continente"] = df["Nationality"].apply(obter_continente)

print(df["Continente"].value_counts())
print("\nNão mapeados (conferir manualmente):")
print(df.loc[df["Continente"] == "Não mapeado", "Nationality"].unique())

Continente
Europe           10192
South America     3383
Asia              1256
Africa            1093
North America      860
Não mapeado        535
Oceania            267
Desconhecido         2
Name: count, dtype: int64

Não mapeados (conferir manualmente):
<StringArray>
[   'Bosnia Herzegovina',              'DR Congo',        'Korea Republic',
         'Guinea Bissau',   'São Tomé & Príncipe',               'Curacao',
         'FYR Macedonia', 'FIFA16_NationName_215',  'Central African Rep.',
        'St Kitts Nevis',     'Trinidad & Tobago',              'China PR',
        'Chinese Taipei',     'Antigua & Barbuda',             'Korea DPR',
           'Timor-Leste',              'St Lucia']
Length: 17, dtype: str


## 16. Agregações (a "tabela dinâmica")

In [24]:
top_paises = df["Nationality"].value_counts().head(10).index

# Pergunta 1: Rating médio por país (top 10 em nº de jogadores)
rating_por_pais = (
    df[df["Nationality"].isin(top_paises)]
    .groupby("Nationality")["Rating"]
    .agg(media="mean", n_jogadores="count")
    .sort_values("media", ascending=False)
)
print(rating_por_pais)

# Pergunta 2: Rating médio por idade (para achar o pico de desempenho)
rating_por_idade = df.groupby("Age")["Rating"].mean()
print(rating_por_idade)

# Pergunta 3: pivot país x posição (Rating médio)
pivot_pais_posicao = pd.pivot_table(
    df[df["Nationality"].isin(top_paises)],
    values="Rating", index="Nationality", columns="Grupo_Posicao", aggfunc="mean"
).round(1)
print(pivot_pais_posicao)

# Pergunta 5: clubes com mais jogadores elite (excluindo o free agent)
elite_por_clube = (
    df[df["Elite"] & (df["Club_Position"] != "Free Agent")]
    .groupby("Club").size()
    .sort_values(ascending=False)
    .head(10)
)
print(elite_por_clube)

# Pergunta 6: oportunidades de contrato acabando entre jogadores bons (Rating >= 75)
oportunidades = df[df["Contrato_Acabando"] & (df["Rating"] >= 75)][
    ["Name", "Age", "Club", "Rating", "Contract_Expiry", "Grupo_Posicao"]
].sort_values("Rating", ascending=False)
print(oportunidades.head(15))

                         media  n_jogadores
Nationality                                
Spain                70.296627         1008
Brazil               70.023887          921
Germany              68.130624          689
Argentina            67.532361         1097
Italy                67.528628          751
France               67.507708          973
Colombia             65.305743          592
England              62.867120         1618
Japan                61.794055          471
Republic of Ireland  59.597285          442
Age
17    55.477707
18    56.769231
19    58.572709
20    60.657285
21    62.990803
22    64.406859
23    65.511062
24    67.044753
25    67.767795
26    68.397490
27    68.837743
28    69.280112
29    69.479167
30    69.454225
31    69.956587
32    69.867725
33    69.150502
34    69.170347
35    69.381356
36    69.295597
37    68.970588
38    66.500000
39    68.750000
40    67.812500
41    64.000000
42    64.000000
43    60.333333
44    67.333333
47    45.000000
Name

## 18. Completar o mapeamento de continente

In [26]:
print(df[df["Nationality"] == "FIFA16_NationName_215"][["Name", "Nationality"]])

                       Name            Nationality
3185          Wesley Lautoa  FIFA16_NationName_215
11297  Georges Gope-Fenepej  FIFA16_NationName_215


In [27]:
mapa_extra = {
    "Bosnia Herzegovina": "Europe",
    "FYR Macedonia": "Europe",
    "DR Congo": "Africa",
    "Guinea Bissau": "Africa",
    "Central African Rep.": "Africa",
    "Korea Republic": "Asia",
    "Korea DPR": "Asia",
    "China PR": "Asia",
    "Chinese Taipei": "Asia",
    "Timor-Leste": "Asia",
    "São Tomé & Príncipe": "Africa",
    "Curacao": "South America",   # ilha caribenha, mas convenção FIFA a coloca na confederação da CONCACAF -> América
    "St Kitts Nevis": "South America",
    "Trinidad & Tobago": "South America",
    "Antigua & Barbuda": "South America",
    "St Lucia": "South America",
    "FIFA16_NationName_215": "Desconhecido",  # valor de placeholder, não é país
}

def obter_continente_v2(pais):
    if pais in excecoes_reino_unido:
        return excecoes_reino_unido[pais]
    if pais in mapa_extra:
        return mapa_extra[pais]
    try:
        codigo = pc.country_name_to_country_alpha2(pais)
        continente_cod = pc.country_alpha2_to_continent_code(codigo)
        return pc.convert_continent_code_to_continent_name(continente_cod)
    except Exception:
        return "Não mapeado"

df["Continente"] = df["Nationality"].apply(obter_continente_v2)

print(df["Continente"].value_counts())
print("Ainda não mapeados:", df.loc[df["Continente"]=="Não mapeado", "Nationality"].unique())

Continente
Europe           10262
South America     3409
Asia              1611
Africa            1175
North America      860
Oceania            267
Desconhecido         4
Name: count, dtype: int64
Ainda não mapeados: <StringArray>
[]
Length: 0, dtype: str


## 19. Salvar dados tratados

In [29]:
df.to_csv("fifa17_tratado.csv", index=False)
print("Salvo:", len(df), "linhas x", df.shape[1], "colunas")

Salvo: 17588 linhas x 63 colunas
